## MAGIC Gamma Telescope Classification

The obective of this analysis is to investigate whether logistic regression and SVMs can distinguish between gamma-ray and hadron events using measurements collected from the MAGIC Gamma Telescope dataset provided [here](https://archive.ics.uci.edu/dataset/159/magic+gamma+telescope).

The target variable is `class`, which identifies whether the observed event is a gamma-ray event (`g`) or a hadron event (`h`) which makes the dataset suitable for binary classification.

## EDA

In [1]:
# Imports of highest import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)
import time

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

RANDOM_STATE = 42

In [2]:
columns = [
    "fLength",
    "fWidth",
    "fSize",
    "fConc",
    "fConc1",
    "fAsym",
    "fM3Long",
    "fM3Trans",
    "fAlpha",
    "fDist",
    "class"
]

df = pd.read_csv(
    "data/magic04.data",
    header=None,
    names=columns
)

print("Dataset dimensions:", df.shape)
print("\nNumber of observations:", df.shape[0])
print("Number of columns:", df.shape[1])
df.head()

Dataset dimensions: (19020, 11)

Number of observations: 19020
Number of columns: 11


,fLength,fWidth,fSize,fConc,fConc1,fAsym,fM3Long,fM3Trans,fAlpha,fDist,class
0,28.7967,16.0021,2.6449,0.3918,0.1982,27.7004,22.0110,-8.2027,40.0920,81.8828,g
1,31.6036,11.7235,2.5185,0.5303,0.3773,26.2722,23.8238,-9.9574,6.3609,205.2610,g
2,162.0520,136.0310,4.0612,0.0374,0.0187,116.7410,-64.8580,-45.2160,76.9600,256.7880,g
3,23.8172,9.5728,2.3385,0.6147,0.3922,27.2107,-6.4633,-7.1513,10.4490,116.7370,g
4,75.1362,30.9205,3.1611,0.3168,0.1832,-5.5277,28.5525,21.8393,4.6480,356.4620,g


In [3]:
print(df["class"].value_counts())

print("\nClass proportions:")
print(df["class"].value_counts(normalize=True) * 100)
df.info()
df.isnull().sum()
df.describe().T

class
g    12332
h     6688
Name: count, dtype: int64

Class proportions:
class
g    64.837014
h    35.162986
Name: proportion, dtype: float64
<class 'pandas.DataFrame'>
RangeIndex: 19020 entries, 0 to 19019
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   fLength   19020 non-null  float64
 1   fWidth    19020 non-null  float64
 2   fSize     19020 non-null  float64
 3   fConc     19020 non-null  float64
 4   fConc1    19020 non-null  float64
 5   fAsym     19020 non-null  float64
 6   fM3Long   19020 non-null  float64
 7   fM3Trans  19020 non-null  float64
 8   fAlpha    19020 non-null  float64
 9   fDist     19020 non-null  float64
 10  class     19020 non-null  str    
dtypes: float64(10), str(1)
memory usage: 1.6 MB


,count,mean,std,min,25%,50%,75%,max
fLength,19020.0,53.250154,42.364855,4.2835,24.336000,37.14770,70.122175,334.1770
fWidth,19020.0,22.180966,18.346056,0.0000,11.863800,17.13990,24.739475,256.3820
fSize,19020.0,2.825017,0.472599,1.9413,2.477100,2.73960,3.101600,5.3233
fConc,19020.0,0.380327,0.182813,0.0131,0.235800,0.35415,0.503700,0.8930
fConc1,19020.0,0.214657,0.110511,0.0003,0.128475,0.19650,0.285225,0.6752
fAsym,19020.0,-4.331745,59.206062,-457.9161,-20.586550,4.01305,24.063700,575.2407
fM3Long,19020.0,10.545545,51.000118,-331.7800,-12.842775,15.31410,35.837800,238.3210
fM3Trans,19020.0,0.249726,20.827439,-205.8947,-10.849375,0.66620,10.946425,179.8510
fAlpha,19020.0,27.645707,26.103621,0.0000,5.547925,17.67950,45.883550,90.0000
fDist,19020.0,193.818026,74.731787,1.2826,142.492250,191.85145,240.563825,495.5610


In [ ]:
features = df.drop(columns="class")
fig, axes = plt.subplots(5, 2, figsize=(15, 20))
for i, column in enumerate(features.columns):
    sns.histplot(
        data=df,
        x=column,
        kde=True,
        ax=axes[i // 2, i % 2]
    )
    axes[i // 2, i % 2].set_title(f"Distribution of {column}")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(15, 22))
for i, column in enumerate(features.columns):
    sns.boxplot(
        data=df,
        x="class",
        y=column,
        hue="class",
        legend=False,
        ax=axes[i // 2, i % 2]
    )
    axes[i // 2, i % 2].set_title(f"{column} by class")
plt.tight_layout()
plt.show()


In [ ]:
correlation_matrix = features.corr()
plt.figure(figsize=(12, 9))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    center=0,
    fmt=".2f"
)
plt.title("Correlation Heatmap of Explanatory Variables")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.scatterplot(
    data=df,
    x="fLength",
    y="fWidth",
    hue="class",
    alpha=0.4,
    s=15,
    ax=axes[0]
)
axes[0].set_title("fLength vs fWidth")
sns.scatterplot(
    data=df,
    x="fAlpha",
    y="fDist",
    hue="class",
    alpha=0.4,
    s=15,
    ax=axes[1]
)
axes[1].set_title("fAlpha vs fDist")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(
    data=df,
    x="fAlpha",
    hue="class",
    fill=True,
    common_norm=False,
    alpha=0.3
)
plt.title("Distribution of fAlpha by Particle Class")
plt.xlabel("fAlpha")
plt.ylabel("Density")
plt.show()
df.groupby("class")[features.columns].median().T

### Observations

Chances are fAlpha will be our most promising indicator, and it should be noted that the strong correlation between some variables may indicate some overlapping information. The skewed distributions of fLength, fWidth, fSize, and fAlpha may also indicate some extreme observations to keep in mind. There's also a large class imbalance, so a classifier predicting gamma (`g`) for every observation would be 64.84% accurate. This is the minimum we need to beat!

## Logistic Regression

In [9]:
X = df.drop(columns="class")
y = df["class"]

# 80/20 test split, seed is 42
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)
print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))
print("\nTraining class proportions:")
print(y_train.value_counts(normalize=True))
print("\nTesting class proportions:")
print(y_test.value_counts(normalize=True))

Training observations: 15216
Testing observations: 3804

Training class proportions:
class
g    0.648396
h    0.351604
Name: proportion, dtype: float64

Testing class proportions:
class
g    0.648265
h    0.351735
Name: proportion, dtype: float64


In [10]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000))
])

# Keep track of training runtime
start_time = time.perf_counter()
logistic_model.fit(X_train, y_train)
logistic_training_time = time.perf_counter() - start_time

# Generate predictions
y_pred = logistic_model.predict(X_test)

# Calculate prediction probabilities
y_prob = logistic_model.predict_proba(X_test)[:, 1]

# Evaluate performance
print("Logistic Regression Results")
print("---------------------------")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, pos_label="g"))
print("Recall:", recall_score(y_test, y_pred, pos_label="g"))
print("F1-score:", f1_score(y_test, y_pred, pos_label="g"))
print("ROC-AUC:", roc_auc_score(y_test == "g", y_prob))
gamma_index = list(logistic_model.classes_).index("g")
y_prob = logistic_model.predict_proba(X_test)[:, gamma_index]
# Calculate ROC-AUC using gamma as the positive class
auc = roc_auc_score(
    (y_test == "g").astype(int),
    y_prob
)
print("Corrected ROC-AUC:", auc)

print("Training time:", logistic_training_time, "seconds")

Logistic Regression Results
---------------------------
Accuracy: 0.7991587802313355
Precision: 0.8087808417997098
Recall: 0.9038929440389294
F1-score: 0.8536959019532746
ROC-AUC: 0.1530991893336825
Corrected ROC-AUC: 0.8469008106663175
Training time: 0.04024179931730032 seconds


In [ ]:
ConfusionMatrixDisplay.from_estimator(
    logistic_model,
    X_test,
    y_test,
    cmap="Blues"
)

plt.title("Logistic Regression Confusion Matrix")
plt.show()

In [19]:
# Extract logistic regression coefficients
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": logistic_model.named_steps["classifier"].coef_[0]
})

# Sort by absolute coefficient magnitude
coefficients["Absolute Coefficient"] = (
    coefficients["Coefficient"].abs()
)
coefficients = coefficients.sort_values(
    "Absolute Coefficient",
    ascending=False
)
print(coefficients)

# plt.figure(figsize=(10, 6))
# sns.barplot(
#     data=coefficients,
#     x="Coefficient",
#     y="Feature",
#     hue="Coefficient",
#     palette="coolwarm",
#     legend=False
# )
# plt.axvline(0, color="black", linewidth=0.8)
# plt.title("Logistic Regression Feature Coefficients")
# plt.xlabel("Standardized Regression Coefficient")
# plt.ylabel("Feature")
# plt.tight_layout()
# plt.show()

    Feature  Coefficient  Absolute Coefficient
0   fLength     1.201220              1.201220
8    fAlpha     1.167837              1.167837
4    fConc1     0.555258              0.555258
6   fM3Long    -0.334168              0.334168
2     fSize     0.301297              0.301297
1    fWidth     0.109836              0.109836
9     fDist     0.046349              0.046349
7  fM3Trans    -0.025791              0.025791
3     fConc     0.018162              0.018162
5     fAsym    -0.016118              0.016118


In [21]:
print(classification_report(y_test, y_pred))
# RocCurveDisplay.from_predictions(
#     (y_test == "g").astype(int),
#     y_prob
# )

# plt.title("Logistic Regression ROC Curve")
# plt.plot([0, 1], [0, 1], linestyle="--", color="grey")
# plt.show()

              precision    recall  f1-score   support

           g       0.81      0.90      0.85      2466
           h       0.77      0.61      0.68      1338

    accuracy                           0.80      3804
   macro avg       0.79      0.76      0.77      3804
weighted avg       0.80      0.80      0.79      3804



### Some observations

The initial results suggest that fLength and fAlpha are the most important predictors with fConc1, fM3Long, and fSize providing some additional information.

The model has high gamma recall but low hadron recall, which indicates that it's able to identify most gamma observations but misses quite a few hadron observations. We'll have to investigate class_weight='balanced' later.

## Linear SVM

In [14]:
# Create Linear SVM pipeline
linear_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LinearSVC(
        C=1.0,
        max_iter=10000,
        random_state=RANDOM_STATE
    ))
])

# Train and measure runtime
start_time = time.perf_counter()

linear_svm.fit(X_train, y_train)
linear_svm_training_time = time.perf_counter() - start_time
svm_pred = linear_svm.predict(X_test)
svm_scores = linear_svm.decision_function(X_test)

# LinearSVC's positive decision scores correspond to classes_[1].
# Our desired positive class is gamma, classes_[0].
gamma_scores = -svm_scores

# Calculate metrics
print("Linear SVM Results")
print("------------------")
print("Accuracy:", accuracy_score(y_test, svm_pred))
print("Precision:", precision_score(
    y_test, svm_pred, pos_label="g"
))
print("Recall:", recall_score(
    y_test, svm_pred, pos_label="g"
))
print("F1-score:", f1_score(
    y_test, svm_pred, pos_label="g"
))
print("ROC-AUC:", roc_auc_score(
    (y_test == "g").astype(int),
    gamma_scores
))
print("Training time:", linear_svm_training_time, "seconds")

Linear SVM Results
------------------
Accuracy: 0.7981072555205048
Precision: 0.8062770562770563
Recall: 0.9063260340632603
F1-score: 0.8533791523482245
ROC-AUC: 0.8470556822411099
Training time: 0.037301200442016125 seconds


In [22]:
# ConfusionMatrixDisplay.from_estimator(
#     linear_svm,
#     X_test,
#     y_test,
#     cmap="Blues"
# )

# plt.title("Linear SVM Confusion Matrix")
# plt.show()

print(classification_report(y_test, svm_pred))

              precision    recall  f1-score   support

           g       0.81      0.91      0.85      2466
           h       0.78      0.60      0.68      1338

    accuracy                           0.80      3804
   macro avg       0.79      0.75      0.76      3804
weighted avg       0.80      0.80      0.79      3804



In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

RocCurveDisplay.from_predictions(
    (y_test == "g").astype(int),
    y_prob,
    name="Logistic Regression",
    ax=ax
)

RocCurveDisplay.from_predictions(
    (y_test == "g").astype(int),
    gamma_scores,
    name="Linear SVM",
    ax=ax
)

ax.plot([0, 1], [0, 1], "--", color="grey")

plt.title("ROC Curve Comparison")
plt.show()

In [18]:
results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, pos_label="g"),
        "Recall": recall_score(y_test, y_pred, pos_label="g"),
        "F1-score": f1_score(y_test, y_pred, pos_label="g"),
        "ROC-AUC": roc_auc_score(
            (y_test == "g").astype(int), y_prob
        ),
        "Training Time": logistic_training_time
    },
    {
        "Model": "Linear SVM",
        "Accuracy": accuracy_score(y_test, svm_pred),
        "Precision": precision_score(y_test, svm_pred, pos_label="g"),
        "Recall": recall_score(y_test, svm_pred, pos_label="g"),
        "F1-score": f1_score(y_test, svm_pred, pos_label="g"),
        "ROC-AUC": roc_auc_score(
            (y_test == "g").astype(int), gamma_scores
        ),
        "Training Time": linear_svm_training_time
    }
])

results.round(4)

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC,Training Time
0,Logistic Regression,0.7992,0.8088,0.9039,0.8537,0.8469,0.0402
1,Linear SVM,0.7981,0.8063,0.9063,0.8534,0.8471,0.0373


## Class weight Investigation

As we've discovered, the classes are not well balanced at all, so lets check how changing class_weight to be balanced affects things. My assumption is that we'll see improved metrics for hadron observations at the expense of gamma observations.

In [24]:
logistic_balanced = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

logistic_balanced.fit(X_train, y_train)

balanced_pred = logistic_balanced.predict(X_test)

print(classification_report(y_test, balanced_pred))

              precision    recall  f1-score   support

           g       0.85      0.83      0.84      2466
           h       0.70      0.73      0.71      1338

    accuracy                           0.79      3804
   macro avg       0.77      0.78      0.78      3804
weighted avg       0.80      0.79      0.79      3804



## Polynomial SVMs

We used a linear SVM earlier, but I wonder how overfitting these things with more complex polynomials would work out. I'm sure we'll see improved scores in the training data but the tests will be worse off.

In [34]:
# Polynomial SVM model
poly_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(
        kernel="poly",
        degree=3,
        C=1.0,
        gamma="scale",
        class_weight=None
    ))
])

start_time = time.perf_counter()

poly_svm.fit(X_train, y_train)
poly_training_time = time.perf_counter() - start_time
poly_pred = poly_svm.predict(X_test)

# SVC decision scores are positive for hadron observations
poly_scores = -poly_svm.decision_function(X_test)

print("Polynomial SVM Results")
print("----------------------")
print("Accuracy:", accuracy_score(y_test, poly_pred))
print("Precision:", precision_score(
    y_test, poly_pred, pos_label="g"
))
print("Recall:", recall_score(
    y_test, poly_pred, pos_label="g"
))
print("F1-score:", f1_score(
    y_test, poly_pred, pos_label="g"
))
print("ROC-AUC:", roc_auc_score(
    (y_test == "g").astype(int),
    poly_scores
))
print("Training time:", poly_training_time)
print("----------------------")
print(classification_report(
    y_test,
    poly_pred
))

Polynomial SVM Results
----------------------
Accuracy: 0.812565720294427
Precision: 0.7859706362153345
Recall: 0.9768856447688564
F1-score: 0.8710902187669499
ROC-AUC: 0.8819466417417384
Training time: 3.368228300474584
----------------------
              precision    recall  f1-score   support

           g       0.79      0.98      0.87      2466
           h       0.92      0.51      0.66      1338

    accuracy                           0.81      3804
   macro avg       0.85      0.74      0.76      3804
weighted avg       0.83      0.81      0.80      3804



In [35]:
# RBF SBF Model
rbf_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight=None
    ))
])

start_time = time.perf_counter()

rbf_svm.fit(X_train, y_train)
rbf_training_time = time.perf_counter() - start_time
rbf_pred = rbf_svm.predict(X_test)
rbf_scores = -rbf_svm.decision_function(X_test)

print("RBF SVM Results")
print("----------------")
print("Accuracy:", accuracy_score(y_test, rbf_pred))
print("Precision:", precision_score(
    y_test, rbf_pred, pos_label="g"
))
print("Recall:", recall_score(
    y_test, rbf_pred, pos_label="g"
))
print("F1-score:", f1_score(
    y_test, rbf_pred, pos_label="g"
))
print("ROC-AUC:", roc_auc_score(
    (y_test == "g").astype(int),
    rbf_scores
))
print("Training time:", rbf_training_time)
print("----------------------")
print(classification_report(
    y_test,
    rbf_pred
))

RBF SVM Results
----------------
Accuracy: 0.8730283911671924
Precision: 0.8619934282584885
Recall: 0.9574209245742092
F1-score: 0.9072046109510087
ROC-AUC: 0.9223135691745558
Training time: 2.7806192999705672
----------------------
              precision    recall  f1-score   support

           g       0.86      0.96      0.91      2466
           h       0.90      0.72      0.80      1338

    accuracy                           0.87      3804
   macro avg       0.88      0.84      0.85      3804
weighted avg       0.88      0.87      0.87      3804



### Observations

It appears just from looking at the numbers that the models have improved their performance, but the runtime has gone up significantly compared to the linear SVM and logistic regression models. However, let's check the balanced class weights with the RBF SVMs to see if that improves what appears to be the best-performing model.

## RBF SVM Class Weight Check

In [32]:
rbf_svm_balanced = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced"
    ))
])

rbf_svm_balanced.fit(X_train, y_train)

rbf_balanced_pred = rbf_svm_balanced.predict(X_test)

print(classification_report(
    y_test,
    rbf_balanced_pred
))

              precision    recall  f1-score   support

           g       0.90      0.90      0.90      2466
           h       0.82      0.81      0.81      1338

    accuracy                           0.87      3804
   macro avg       0.86      0.85      0.86      3804
weighted avg       0.87      0.87      0.87      3804



### Observations

It appears just from looking at the numbers that the models improved their score, but the runtime has gone up significantly for both of these models and the improvement wasn't too much higher.

## Overall Results Comparison Table

In [36]:
models = {
    "Logistic Regression": (logistic_model, y_pred),
    "Logistic Regression Balanced": (logistic_balanced, balanced_pred),
    "Linear SVM": (linear_svm, svm_pred),
    "Polynomial SVM": (poly_svm, poly_pred),
    "RBF SVM": (rbf_svm, rbf_pred),
    "RBF SVM Balanced": (rbf_svm_balanced, rbf_balanced_pred)
}

model_results = []

for name, (model, predictions) in models.items():

    scores = -model.decision_function(X_test)

    model_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, pos_label="g"
        ),
        "Recall": recall_score(
            y_test, predictions, pos_label="g"
        ),
        "F1-score": f1_score(
            y_test, predictions, pos_label="g"
        ),
        "Macro F1": f1_score(
            y_test, predictions, average="macro"
        ),
        "ROC-AUC": roc_auc_score(
            (y_test == "g").astype(int), scores
        )
    })

comparison = pd.DataFrame(model_results)

comparison.round(4)

,Model,Accuracy,Precision,Recall,F1-score,Macro F1,ROC-AUC
0,Logistic Regression,0.7992,0.8088,0.9039,0.8537,0.7667,0.8469
1,Logistic Regression Balanced,0.7936,0.8512,0.8260,0.8384,0.7764,0.8496
2,Linear SVM,0.7981,0.8063,0.9063,0.8534,0.7647,0.8471
3,Polynomial SVM,0.8126,0.7860,0.9769,0.8711,0.7639,0.8819
4,RBF SVM,0.8730,0.8620,0.9574,0.9072,0.8531,0.9223
5,RBF SVM Balanced,0.8688,0.8961,0.9023,0.8992,0.8558,0.9245


In [47]:
comparison_melted = comparison.melt(
    id_vars="Model",
    value_vars=[
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "Macro F1",
        "ROC-AUC"
    ],
    var_name="Metric",
    value_name="Score"
)

# plt.figure(figsize=(13, 6))

# sns.barplot(
#     data=comparison_melted,
#     x="Metric",
#     y="Score",
#     hue="Model"
# )

# plt.ylim(0, 1)
# plt.title("Classification Performance Comparison")
# plt.tight_layout()
# plt.show()

## Cross Validation and Hyperparameters

At the moment, we're only evaluating with one split. Lets introduce cross validation and see how consistent the performance is. We shoudl also check if we're beholden to different hyperparameters.

In [45]:
from sklearn.model_selection import (
    cross_validate,
    StratifiedKFold
)

# Define the cross-validation strategy
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Models to evaluate
models = {
    "Logistic Regression": logistic_model,
    "Logistic Regression Balanced": logistic_balanced,
    "Linear SVM": linear_svm,
    "Polynomial SVM": poly_svm,
    "RBF SVM": rbf_svm,
    "RBF SVM Balanced": rbf_svm_balanced
}

# Metrics to evaluate
scoring = {
    "accuracy": "accuracy",
    "precision": "precision_macro",
    "recall": "recall_macro",
    "f1": "f1_macro",
    "roc_auc": "roc_auc"
}

cv_results = []

for name, model in models.items():

    start_time = time.perf_counter()

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    training_time = time.perf_counter() - start_time

    result = {
        "Model": name,
        "Macro F1 Mean": scores["test_f1"].mean(),
        "Macro F1 Std": scores["test_f1"].std(),
        "Training Time": training_time,
        "Accuracy Mean": scores["test_accuracy"].mean(),
        "Accuracy Std": scores["test_accuracy"].std(),
        "Precision Mean": scores["test_precision"].mean(),
        "Recall Mean": scores["test_recall"].mean(),
        "ROC-AUC Mean": scores["test_roc_auc"].mean(),
        "ROC-AUC Std": scores["test_roc_auc"].std()
    }

    cv_results.append(result)

# Create comparison DataFrame
cv_comparison = pd.DataFrame(cv_results)

cv_comparison = cv_comparison.sort_values(
    by="Macro F1 Mean",
    ascending=False
)

cv_comparison.round(4)

,Model,Macro F1 Mean,Macro F1 Std,Training Time,Accuracy Mean,Accuracy Std,Precision Mean,Recall Mean,ROC-AUC Mean,ROC-AUC Std
5,RBF SVM Balanced,0.8456,0.0043,4.3245,0.8596,0.0035,0.8466,0.8447,0.9162,0.0037
4,RBF SVM,0.8445,0.0044,3.5084,0.8662,0.0036,0.8755,0.8284,0.9148,0.0040
1,Logistic Regression Balanced,0.7652,0.0074,0.1169,0.7835,0.0073,0.7628,0.7683,0.8390,0.0055
0,Logistic Regression,0.7527,0.0052,0.1161,0.7878,0.0053,0.7781,0.7413,0.8365,0.0055
2,Linear SVM,0.7510,0.0058,0.1142,0.7868,0.0059,0.7776,0.7393,0.8372,0.0054
3,Polynomial SVM,0.7500,0.0052,4.0037,0.8024,0.0041,0.8411,0.7309,0.8700,0.0058


In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    data=cv_comparison,
    x="Model",
    y="Macro F1 Mean"
)

plt.errorbar(
    x=range(len(cv_comparison)),
    y=cv_comparison["Macro F1 Mean"],
    yerr=cv_comparison["Macro F1 Std"],
    fmt="none",
    color="black",
    capsize=5
)

plt.xticks(rotation=30, ha="right")
plt.ylim(0, 1)
plt.ylabel("Cross-Validation Macro F1")
plt.title("Model Comparison - 5-Fold Cross Validation")
plt.tight_layout()
plt.show()